In [19]:
!pip install -q -U mlflow boto3 awscli dagshub optuna xgboost imbalanced-learn

In [20]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

try:
    
    access_key = user_secrets.get_secret("AWS_ACCESS_KEY")
    secret_key = user_secrets.get_secret("AWS_SECRET_KEY")
    
    # Set environment variables
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
    
    print("✅ AWS Credentials successfully loaded!")
    print(f"Access Key ID: {access_key[:4]}****{access_key[-4:]}")
    print(f"Region: {os.environ['AWS_DEFAULT_REGION']}")

except Exception as e:
    print("❌ Failed to find secrets. Make sure you added them in Add-ons -> Secrets.") 

✅ AWS Credentials successfully loaded!
Access Key ID: AKIA****LQZB
Region: us-east-1


In [21]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [22]:
## Setup Mlflow Server
mlflow.set_tracking_uri("http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/")
# Set an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning")

2026/05/13 17:59:22 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://amzn-mlflow-bucket-26/5', creation_time=1778695162329, experiment_id='5', last_update_time=1778695162329, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [23]:
df = pd.read_csv('/kaggle/input/datasets/arnabnath1028/reddit-data/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [30]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

ngram_range = (1, 3)  # Trigram setting
max_features = 10000  # Set max_features to 1000 for TF-IDF

# Step 4: Train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

# Step 2: Vectorization using TF-IDF, fit on training data only
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
X_test_vec = vectorizer.transform(X_test)  # Transform test data

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for XGBoost
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 600)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)

    # Add regularization parameters to prevent "fake" accuracy
    gamma = trial.suggest_float('gamma', 0, 0.5)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)

    model = XGBClassifier(
        n_estimators=n_estimators, 
        learning_rate=learning_rate, 
        max_depth=max_depth, 
        random_state=42, 
        gamma=gamma, 
        subsample=subsample,
        tree_method='hist', 
        device='cuda'
    )
    return accuracy_score(y_test, model.fit(X_train_vec, y_train).predict(X_test_vec))


# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_xgboost, n_trials=100)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = XGBClassifier(
        n_estimators=best_params['n_estimators'], 
        learning_rate=best_params['learning_rate'], 
        max_depth=best_params['max_depth'], 
        random_state=42
    )

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("XGBoost", best_model, X_train_vec, X_test_vec, y_train, y_test)
    return best_model
# Run the experiment for XGBoost
run_optuna_experiment()

[I 2026-05-13 18:30:10,190] A new study created in memory with name: no-name-04efb35e-8d4b-4261-9acf-b7587e496089
[I 2026-05-13 18:30:12,001] Trial 0 finished with value: 0.5148689072672884 and parameters: {'n_estimators': 435, 'learning_rate': 0.07332606195970852, 'max_depth': 9, 'gamma': 0.12711605800879267, 'subsample': 0.6168602336539123}. Best is trial 0 with value: 0.5148689072672884.
[I 2026-05-13 18:30:22,469] Trial 1 finished with value: 0.6600387119479149 and parameters: {'n_estimators': 322, 'learning_rate': 0.007693229741185111, 'max_depth': 12, 'gamma': 0.07776642760956698, 'subsample': 0.7445873787161438}. Best is trial 1 with value: 0.6600387119479149.
[I 2026-05-13 18:30:29,341] Trial 2 finished with value: 0.5057188104874186 and parameters: {'n_estimators': 494, 'learning_rate': 0.007857444583266471, 'max_depth': 6, 'gamma': 0.05656011510693015, 'subsample': 0.9334967989601738}. Best is trial 1 with value: 0.6600387119479149.
[I 2026-05-13 18:30:40,368] Trial 3 finishe

🏃 View run XGBoost_SMOTE_TFIDF_Trigrams at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/5/runs/4c3c2b2f30914d9fa7e3693f48d3d30d
🧪 View experiment at: http://ec2-44-201-225-146.compute-1.amazonaws.com:5000/#/experiments/5


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.015002639635867732, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=15, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=468, n_jobs=None,
              num_parallel_tree=None, ...)